In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
import torch

text = open('input.txt', encoding='utf-8').read()
chars = sorted(set(text))                      # 문자 단위 vocab (약 65개)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, B=16, T=64):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - T - 1, (B,))          # 랜덤 시작점 B개
    x = torch.stack([d[i:i+T] for i in ix])           # 입력  (B, T)
    y = torch.stack([d[i+1:i+T+1] for i in ix])       # 정답  (B, T), 한 칸 밀림
    return x, y

In [ ]:
x, y = get_batch('train')
print(x.shape, y.shape)          # torch.Size([16, 64]) 둘 다
print(repr(decode(x[0, :20].tolist())))
print(repr(decode(y[0, :20].tolist())))   # 한 글자씩 밀려 있어야 함

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, max_T=256):
        super().__init__()
        assert d % h == 0
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)   # W_Q, W_K, W_V를 한 행렬로
        self.wo = nn.Linear(d, d, bias=False)        # W_O
        mask = torch.tril(torch.ones(max_T, max_T, dtype=torch.bool))
        self.register_buffer('mask', mask)           # 학습 안 되는 상수

    def forward(self, x, mask):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)                          # 각 (B, T, d)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2)
                   for t in (q, k, v)]                                   # (B, h, T, dk)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)           # (B, h, T, T)
        scores = scores.masked_fill(~mask.to(x.device), float('-inf'))  # 미래 칸 막기
        w = F.softmax(scores, dim=-1)
        self.last_w = w.detach()
        out = (w @ v).transpose(1, 2).reshape(B, T, d)                  # head concat
        return self.wo(out)

In [ ]:
torch.manual_seed(0)
attn = MultiHeadAttention(d=32, h=4)
x = torch.randn(1, 8, 32)
y1 = attn(x)

x2 = x.clone()
x2[0, 5] += 1.0                     # 5번 위치만 바꿈
y2 = attn(x2)
print((y1 - y2).abs().sum(-1))      # 위치별 출력 변화량

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, max_T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h, max_T)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # 소통
        x = x + self.ffn(self.ln2(x))      # 처리
        return x


class TinyGPT(nn.Module):
    def __init__(self, V, d=128, h=4, n_layer=4, max_T=256):
        super().__init__()
        self.max_T = max_T
        self.tok_emb = nn.Embedding(V, d)
        self.pos_emb = nn.Embedding(max_T, d)           # 학습형 위치 embedding
        self.blocks = nn.ModuleList([Block(d, h, max_T) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)       # (B,T,d) + (T,d)
        for blk in self.blocks:
            x = blk(x, causal_mask(T))
        logits = self.head(self.ln_f(x))                # (B, T, V)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

In [ ]:
model = TinyGPT(len(chars))
print(sum(p.numel() for p in model.parameters()))   # 전체 파라미터 수

x, y = get_batch('train')
logits, loss = model(x, y)
print(logits.shape, loss.item())    # (16, 64, 65), 그리고 loss는?

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TinyGPT(len(chars)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

@torch.no_grad()
def estimate_loss(iters=50):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = []
        for _ in range(iters):
            x, y = get_batch(split)
            losses.append(model(x.to(device), y.to(device))[1].item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

for step in range(3000):
    x, y = get_batch('train')
    _, loss = model(x.to(device), y.to(device))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 300 == 0:
        print(step, estimate_loss())

In [ ]:
@torch.no_grad()
def generate(model, idx, n_new, temperature=1.5):
    for _ in range(n_new):
        idx_cond = idx[:, -model.max_T:]            # 최대 길이를 넘지 않게 자름
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature     # 마지막 위치의 예측만 사용
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)            # 확률대로 한 글자 뽑기
        idx = torch.cat([idx, nxt], dim=1)           # 입력 끝에 붙이고 반복
    return idx

start = torch.zeros((1, 1), dtype=torch.long, device=device)   # ID 0 = '\n'
print(decode(generate(model, start, 300)[0].tolist()))

In [ ]:
def causal_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))

def bidirectional_mask(T):
    return torch.ones(T, T, dtype=torch.bool)

def prefix_lm_mask(T, n_prefix):
    m = causal_mask(T)
    m[:n_prefix, :n_prefix] = True      # prefix 블록만 양방향으로 풀어 줌
    return m

print(prefix_lm_mask(4, 2).int())       # 6단계에서 그린 표와 같은지 확인

In [ ]:
import matplotlib.pyplot as plt

T = 8
masks = {'causal': causal_mask(T),
         'bidirectional': bidirectional_mask(T),
         'prefix-LM (4)': prefix_lm_mask(T, 4)}

torch.manual_seed(0)
attn = MultiHeadAttention(d=32, h=4)
x = torch.randn(1, T, 32)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, m) in zip(axes, masks.items()):
    attn(x, m)
    ax.imshow(attn.last_w[0, 0])      # 배치 0, head 0의 (T, T) 가중치
    ax.set_title(name)
    ax.set_xlabel('key (주는 쪽)'); ax.set_ylabel('query (받는 쪽)')
plt.show()